In [1]:
import torch
from torch.amp import autocast
import time
import matplotlib.pyplot as plt
from v1_model import ContinuousMotionModel
from dataset.dataset import *
import utils.utils as utils
import soundfile as sf
from IPython.display import clear_output

device = utils.get_device()

In [2]:
# Get the newest model from the directory. That means find the newest folder, and then the newest file in that folder.
model_path = utils.get_latest_model_path("v1_models")
print(f"Model path: {model_path}")

# Load the model
model: ContinuousMotionModel = ContinuousMotionModel.load_model(model_path,device)
model.condition_mask_probabilty = 0.0  # Disable condition mask probability for inference
model = model.to(device)

num_params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters in the model: {num_params}")

Model path: v1_models\first_tests_2025-06-15_04-59-20\first_tests_2025-06-15_04-59-20_epoch_991.pth
Number of parameters in the model: 8294934


In [3]:
import utils.animation.visualisation.new.animation_visualisation as animation_visualisation
print(animation_visualisation.init_visualization(display=False))

http://localhost:8000/utils/animation/visualisation/new/animation_viewer4.html?wsport=8700


In [47]:
with autocast(device_type=device.type, dtype=torch.bfloat16):
    dataset = GPUDataset(
        consolidated_file="dataset/genea2023_dataset/val/main-agent/consolidated.npz",
        seq_length=100,
        seed_length=0,
        batch_size=1,
        epoch_length=1,  # Set to 1 for testing purposes
        return_audio_frame_index=True,  # Set to True to return the audio frame index
    )

    # Use no gradient calculation for inference
    with torch.no_grad():

        gesture_sequence, _, _, main_agent_id_one_hot, start_frames = [
            item.to(device) for item in next(iter(dataset))
        ]
        full_audio_features = dataset.audio.to(device)
        start_frame = start_frames[0].item()  # Extract the first element from the tensor

        # Decode the output using the autoencoder model
        encoded_gesture_sequence = model.pose_encoder.encode(gesture_sequence)

        iteration_counter = 0
        
        while True:
            iteration_counter += 1
            # Start time for the current frame
            frame_start_time = time.time()

            # The audio features also have to be shifted by one frame
            # I have the full audio features and the starting frame, so I extract the audio features for the current frame
            actual_audio_features = full_audio_features[start_frame + iteration_counter: start_frame + iteration_counter + dataset.seq_length, :].unsqueeze(0)

            encoded_gesture_sequence, noisy_gesture_sequence = model.inference(encoded_gesture_sequence, actual_audio_features, main_agent_id_one_hot)

            ########################################################################################################################################################################

            # Decode the output using the autoencoder model
            clear_output(wait=True)
            
            pre_decode_time = time.time()
            unencoded_denoised_frame = model.pose_encoder.decode(encoded_gesture_sequence[:,model.num_of_pre_timestep_frames,:].unsqueeze(0))
            post_decode_time = time.time()
            
            print(f"Time taken for decoding in ms: {(post_decode_time - pre_decode_time) * 1000:.2f} ms")

            denmormalized_unencoded_denoised_frame = dataset.skeleton.denormalize_poses(unencoded_denoised_frame).squeeze(0).squeeze(0)
            animation_visualisation.send_pose(denmormalized_unencoded_denoised_frame.cpu(), dataset.skeleton)
            animation_visualisation.send_debug_tensor(torch.cat((actual_audio_features.squeeze(0).to(torch.float32),noisy_gesture_sequence.squeeze(0).to(torch.float32)), dim=1), "full tensor")


            print(f"Iteration: {iteration_counter}")
            frame_end_time = time.time()
            # Print the time taken for the current frame
            print(f"Frame {iteration_counter} processed in {frame_end_time - frame_start_time:.4f} seconds ({1/(frame_end_time - frame_start_time):.2f} FPS)")

            # Sleep for the remaining time in the 30 FPS frame
            time_to_sleep = max(0, (1/30) - (frame_end_time - frame_start_time) - 0.0005)  # 0.01 is a small buffer to account for processing time
            time.sleep(time_to_sleep)


=== Pose Encoder Profiling Information ===
initialization                : 0.00 ms
preserved_components          : 1.50 ms
auto_encoded_components       : 2.01 ms
denormalization               : 0.00 ms
unhandled_indices             : 0.00 ms
identity_rotations            : 1.00 ms
world_positions               : 2.51 ms
ik_processing                 : 2.00 ms
world_preserve_after_ik       : 1.00 ms
final_normalization           : 1.00 ms
Time taken for decoding in ms: 10.01 ms
Iteration: 33543
Frame 33543 processed in 0.0285 seconds (35.04 FPS)


RuntimeError: Sizes of tensors must match except in dimension 2. Expected size 99 but got size 100 for tensor number 1 in the list.